In [0]:
spark.conf.set(
    "fs.azure.account.key.shadowrevenuedatalake.dfs.core.windows.net",
    "KEY"
)

In [0]:
from pyspark.sql.functions import col, sum as _sum, when

orders_bad = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/orders")
payments_bad = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/payments")
products_bad = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/products").filter("is_current = 1")

orders_good = spark.read.format("delta").load("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/orders_good")
payments_good = spark.read.format("delta").load("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/payments_good")
products_good = spark.read.format("delta").load("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/products_good")

In [0]:
product_price_bad = products_bad.select("product_id", col("price").alias("catalog_price"))

fact_revenue_bad = (
    orders_bad.alias("o")
    .join(product_price_bad.alias("p"), "product_id", "left")
    .join(payments_bad.alias("pay"), "order_id", "left")
    .withColumn("calculated_revenue", col("o.quantity") * col("p.catalog_price"))
)

fact_revenue_bad.write.format("delta").mode("overwrite").save("abfss://gold@shadowrevenuedatalake.dfs.core.windows.net/fact_revenue_bad")
print("Bad fact table rows:", fact_revenue_bad.count())

Bad fact table rows: 20400


In [0]:
product_price_good = products_good.select("product_id", col("price").alias("catalog_price"))

fact_revenue_good = (
    orders_good.alias("o")
    .join(product_price_good.alias("p"), "product_id", "left")
    .join(payments_good.alias("pay"), "order_id", "left")
    .withColumn("calculated_revenue", col("o.quantity") * col("p.catalog_price"))
)

fact_revenue_good.write.format("delta").mode("overwrite").save("abfss://gold@shadowrevenuedatalake.dfs.core.windows.net/fact_revenue_good")
print("Good fact table rows:", fact_revenue_good.count())

Good fact table rows: 20000


In [0]:
missing_payment_good = fact_revenue_good.filter(col("payment_amount").isNull())
print("Missing payments (orders with no payment):", missing_payment_good.count())

orphan_payment = payments_good.join(orders_good, "order_id", "left_anti")
print("Orphan payments (payments with no order):", orphan_payment.count())

Missing payments (orders with no payment): 2000
Orphan payments (payments with no order): 600


In [0]:
from pyspark.sql.functions import abs as _abs

price_mismatch = (
    fact_revenue_good
    .withColumn("pct_diff", _abs(col("o.price") - col("p.catalog_price")) / col("p.catalog_price"))
    .filter(col("pct_diff") > 0.05)
)

print("Price mismatches (>5% deviation from catalog):", price_mismatch.count())

Price mismatches (>5% deviation from catalog): 19047


In [0]:
kpi_bad = fact_revenue_bad.agg(
    _sum("calculated_revenue").alias("total_revenue"),
    _sum("payment_amount").alias("total_payment")
).withColumn("version", col("total_revenue")*0+1).withColumnRenamed("version","tag")

kpi_good = fact_revenue_good.agg(
    _sum("calculated_revenue").alias("total_revenue"),
    _sum("payment_amount").alias("total_payment")
)

kpi_bad_final = kpi_bad.drop("tag").withColumn("version", col("total_revenue")*0)

print("BAD -> total_revenue:", kpi_bad.collect()[0]["total_revenue"], " total_payment:", kpi_bad.collect()[0]["total_payment"])
print("GOOD -> total_revenue:", kpi_good.collect()[0]["total_revenue"], " total_payment:", kpi_good.collect()[0]["total_payment"])

BAD -> total_revenue: 65730543.18000072  total_payment: 19450610.200000014
GOOD -> total_revenue: 64379883.9800  total_payment: 19067003.9600


In [0]:
fact_revenue_bad.coalesce(1).write.mode("overwrite").option("header", True).csv("abfss://gold@shadowrevenuedatalake.dfs.core.windows.net/exports/fact_revenue_bad")
fact_revenue_good.coalesce(1).write.mode("overwrite").option("header", True).csv("abfss://gold@shadowrevenuedatalake.dfs.core.windows.net/exports/fact_revenue_good")